In [0]:
%sql
MERGE INTO retail_lakehouse.gold.fact_sales tgt
USING (
    SELECT
        TransactionID,
        CustomerSK,
        ProductSK,
        StoreSK,
        Quantity,
        Amount,
        TxnDate
    FROM (
        SELECT
            s.TransactionID,
            c.CustomerSK,
            p.ProductSK,
            st.StoreSK,
            s.Quantity,
            s.Quantity * p.UnitPrice AS Amount,
            s.TxnDate,
            ROW_NUMBER() OVER (PARTITION BY s.TransactionID ORDER BY c.CustomerSK) AS rn
        FROM retail_lakehouse.silver.sales s
        JOIN retail_lakehouse.gold.dim_customer c
            ON s.CustomerID = c.CustomerID
            AND c.IsActive = TRUE
        JOIN retail_lakehouse.gold.dim_product p
            ON s.ProductID = p.ProductID
        JOIN retail_lakehouse.gold.dim_store st
            ON s.StoreID = st.StoreID
    ) deduped
    WHERE rn = 1
) src
ON tgt.TransactionID = src.TransactionID
WHEN MATCHED THEN
UPDATE SET
    tgt.Quantity = src.Quantity,
    tgt.Amount = src.Amount,
    tgt.TxnDate = src.TxnDate

WHEN NOT MATCHED THEN
INSERT
(
    SalesSK,
    TransactionID,
    CustomerSK,
    ProductSK,
    StoreSK,
    Quantity,
    Amount,
    TxnDate
)
VALUES
(
    monotonically_increasing_id(),
    src.TransactionID,
    src.CustomerSK,
    src.ProductSK,
    src.StoreSK,
    src.Quantity,
    src.Amount,
    src.TxnDate
);